# Omni-AD-30 全量运行 —— DINOv2 ViT-L 主干教程（518 高分辨率版）

工业图像异常检测（PatchCore + DINOv2 ViT-L/14 自监督主干）。本 notebook 从零跑通 30 类训练 / 预测 / 评测，输出官方指标表。

## 先看这里
- **需要 GPU 运行时**：右上角「代码执行程序 → 更改运行时类型 → T4 GPU」（有 A100 更快、更稳过 100ms 时延约束）。
- **数据要在 Google Drive**：`MyDrive/IAD/Omni-AD-30-release.zip`（打包后拖进 Drive，约 5.8GB）。
- **主干 DINOv2 ViT-L/14**（`blocks.6/12/18` 三层，518 输入 → 37×37 网格，特征 3072 维）：自监督 LVD-142M + 4 寄存器 token，对齐 franca 的 3/6/9 相对层位但更深更宽。
- **权重 ~1.1GB 只下 1 次**：首次跑 `scripts/fetch_dinov2.py` 拉官方 `dinov2_vitl14_reg` 权重并前向校验；之后回写 Drive 缓存，重跑免下载。
- 跑完记得把 `work/` 拷回 Drive 持久化，否则会话重置会丢。


## 0. 挂载 Drive + 检查 GPU/显存

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
if torch.cuda.is_available():
    print("显存", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")


## 1. 解压数据（已解压过可跳过；`-oq` 静默覆盖，重跑安全）

In [ ]:
%%bash
mkdir -p /content/data
unzip -oq /content/drive/MyDrive/IAD/Omni-AD-30-release.zip -d /content/data/
ls /content/data/Omni-AD-30-release | wc -l   # 期望 30


## 2. 干净克隆 swin 分支 + 装依赖

每次都 `rm -rf` 重新克隆，保证拿到最新代码（含 DINOv2 主干 + 518 分辨率）。
DINOv2 走 PyTorch 原生推理（`scaled_dot_product_attention` → GPU 上自动 flash/memory-efficient attention），**不装 onnx / onnxruntime**——ViT-L 用 ONNX 反而更慢且导出极慢。


In [ ]:
%%bash
cd /content
rm -rf IAD-Industrial-Anomaly-Detection
git clone -b swin https://github.com/coder-yu-WICK/IAD-Industrial-Anomaly-Detection.git
cd IAD-Industrial-Anomaly-Detection
git log --oneline -1
pip install -q scikit-learn


## 3. 拉 DINOv2 ViT-L/14 权重（~1.1GB，仅 1 次；已有缓存则跳过）

`scripts/fetch_dinov2.py` 会 torch.hub 拉官方 `dinov2_vitl14_reg` 权重 → 键名对齐 → CPU 前向一致性校验 → 打包到 `model/pretrained/dinov2_vitl14.pth`。

拉完回写 Drive 缓存，下次重跑直接从 Drive 恢复，免再下 1.1GB。


In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
mkdir -p model/pretrained /content/drive/MyDrive/IAD

# 有 Drive 缓存则先恢复，跳过 1.1GB 下载
if [ -f /content/drive/MyDrive/IAD/dinov2_vitl14.pth ] && [ ! -f model/pretrained/dinov2_vitl14.pth ]; then
  echo "从 Drive 恢复缓存权重 ..."
  cp /content/drive/MyDrive/IAD/dinov2_vitl14.pth model/pretrained/dinov2_vitl14.pth
fi

# 拉取 + 校验（本地已存在则脚本内部自动跳过）
python scripts/fetch_dinov2.py

# 回写 Drive 持久化
cp model/pretrained/dinov2_vitl14.pth /content/drive/MyDrive/IAD/dinov2_vitl14.pth


## 4.【可选，强烈建议】实测单图推理时延（100ms 硬约束）

先测再训练：主干前向是时延大头。若端到端逼近 100ms，直接换 `dinov2_vitb14_reg`（franca 同级，快 3~4 倍）。


In [ ]:
import sys, time, numpy as np
sys.path.insert(0, "src")
import torch
from PIL import Image
from patchcore import PatchCore

model = PatchCore(device=torch.device('cuda:0'), backbone='dinov2_vitl14',
                  layers=('blocks.6','blocks.12','blocks.18'),
                  input_size=(518,518), crop_size=(518,518),
                  pretrained_path='model/pretrained/dinov2_vitl14.pth')

x = torch.randn(1,3,518,518).cuda()
with torch.inference_mode():
    for _ in range(10): model.backbone(x)      # warmup
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(20): model.backbone(x)
    torch.cuda.synchronize()
dt = (time.perf_counter()-t0)/20
print(f"主干前向(19 层 @518): {dt*1000:.1f} ms  -> 占 100ms 预算 {dt*10:.0f}%")

# 端到端（含 kNN，随机 27k bank 占位，量级接近真实）
bank = np.random.rand(27000, 3072).astype('float32')
model.bank_dict = {'bank':bank,'coreset_indices':None,
    'norm_scale':float(np.sqrt((bank**2).sum(1).mean())),
    'feature_dim':3072,'backbone':'dinov2_vitl14',
    'layers':['blocks.6','blocks.12','blocks.18'],
    'input_size':[518,518],'crop_size':[518,518],'sigma':4.0}
img = Image.fromarray(np.random.randint(0,255,(518,518,3),dtype='uint8'))
for _ in range(2): model.predict(img)
torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(10): model.predict(img)
torch.cuda.synchronize()
print(f"端到端 predict(含 27k bank kNN): {(time.perf_counter()-t0)/10*1000:.1f} ms")
print("判定：端到端 <100ms 且余量够 -> 继续 ViT-L；逼近/超 -> 换 dinov2_vitb14_reg")


## 5. 软链数据 + 生成 manifest

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
mkdir -p data
ln -s /content/data/Omni-AD-30-release data/Omni-AD-30-release
python -u src/data/sample_manifest.py --data-root data/Omni-AD-30-release


## 6. 训练（默认 DINOv2 ViT-L，518，3072 维）

- 权重已在第 3 步 fetch 好；若跳过第 3 步，`train.py` 会报错并提示先跑 fetch。
- 30 类在 T4 上预计 **3~6 小时**（ViT-L 特征提取比 swin 重）；建议先拿 1~2 类试跑确认不 OOM。
- **显存紧张**（T4 16GB 可能吃紧）：每类全量 patch 特征 ≈3.4GB；若 OOM，加 `--max-embed 200000` 子采样。
- **时延/显存超了想换 ViT-B**：先别急，把第 4 步测的时延发我，我帮你切 `dinov2_vitb14_reg`（几乎零改动）。


In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/train.py   --data-root data/Omni-AD-30-release   --manifest data/Omni-AD-30-release/train_manifest.csv   --output-dir work/model_dinov2   --device cuda:0 --seed 2026 --num-workers 4


## 7. 预测（自动识别主干/分辨率，从 ckpt 读，无需指定）

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python -u src/predict.py   --data-root data/Omni-AD-30-release   --manifest data/Omni-AD-30-release/test_manifest.csv   --model-dir work/model_dinov2   --output-dir work/pred_dinov2   --device cuda:0 --num-workers 4


## 8. 评测 → 30 类指标表

In [ ]:
%%bash
cd /content/IAD-Industrial-Anomaly-Detection
python src/evaluate.py   --predictions-dir work/pred_dinov2   --data-root data/Omni-AD-30-release   --manifest data/Omni-AD-30-release/test_manifest.csv


## 附：跑完把产物拷回 Drive（防会话重置丢失）

```bash
cp -r work /content/drive/MyDrive/IAD/work_dinov2
```

> 注意：`data/` 已 gitignore，数据严禁提交 GitHub；代码/commit 里不要出现学校信息。
> 权重 `model/pretrained/dinov2_vitl14.pth` 已在第 3 步回写 Drive（`MyDrive/IAD/dinov2_vitl14.pth`），下次重跑免下载。
